In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
file = "hotel_revenue_data.xlsx"
df = pd.read_excel(file, header=2)

In [ ]:
#-------------------------------------------------------------------------------------
# Q2(a) - Missing Value
#-------------------------------------------------------------------------------------
real_cols = []
target = "Monthly_Revenue_RM"
id_col = "Month_ID"

In [ ]:
for c in df.columns:
    if not c.startswith("Unnamed"):
        real_cols.append(c)

In [ ]:
missing = df[real_cols].isnull().sum()

In [ ]:
print("\n========== Q2(a) - Missing Value ==========")
print(missing)
print("Total missing:", missing.sum())

In [ ]:
# Q2(a) - Correct predictor
#-------------------------------------------------------------------------------------
predictors = []

In [ ]:
for col in df.columns:
    if col not in [target, id_col] and not col.startswith("Unnamed"):
        predictors.append(col)

In [ ]:
print("\n========== Q2(a) - Variable Exclusion ==========")
print("\nPredictors used:", predictors)

In [ ]:
# Q2(a) - Z-score Outlier Detection (exclude Month_ID, Weekend_Promotion, target)
#-------------------------------------------------------------------------------------
zscore_vars = []

In [ ]:
for c in predictors:
    if c != "Weekend_Promotion":
        zscore_vars.append(c)

In [ ]:
z_scores = (df[zscore_vars] - df[zscore_vars].mean()) / df[zscore_vars].std(ddof=1)
z_abs = z_scores.abs()
outlier_mask = (z_abs > 3)
outlier_rows = df[outlier_mask.any(axis=1)]

In [ ]:
print("\n========== Z-score Outlier Detection ==========")
print(f"Total Outlier Rows : {len(outlier_rows)}")

In [ ]:
for index in outlier_rows.index:
    variables = list(outlier_mask.columns[outlier_mask.loc[index]])
    print(f"Month_ID {df.loc[index,'Month_ID']} -> {variables}")

In [ ]:
df_clean = df[~outlier_mask.any(axis=1)].reset_index(drop=True)
print("\n========== After Removing Outliers ==========")
print("Original Sample Size :", len(df))
print("Removed Rows :", len(df) - len(df_clean))
print("New Sample Size :", len(df_clean))

In [ ]:
#-------------------------------------------------------------------------------------
# Q2(b) Multiple Linear Regression
#-------------------------------------------------------------------------------------
X = df_clean[predictors]
y = df_clean[target]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

In [ ]:
coef = model.params
equation = f"{target} = {coef['const']:.4f}"
for variable in predictors:
    equation += f" + ({coef[variable]:.4f})*{variable}"

In [ ]:
print("\n========== Regression Summary ==========")
print(model.summary())

In [ ]:
print("\n========== Estimated Regression Equation ==========")
print(equation)

In [ ]:
#-------------------------------------------------------------------------------------
# Q2(c) F-Test
#-------------------------------------------------------------------------------------
print("\n========== F Test ==========")

In [ ]:
f_statistic = model.fvalue
p_value = model.f_pvalue

In [ ]:
print("F Statistic :", f_statistic)
print("p-value (Significance F) :", p_value)

In [ ]:
alpha = 0.01

In [ ]:
if p_value < alpha:
    print("Reject H0 - Overall regression is statistically significant.")
else:
    print("Fail to Reject H0 - Overall regression is NOT statistically significant.")

In [ ]:
#-------------------------------------------------------------------------------------
# Q2(d) T-Test
#-------------------------------------------------------------------------------------
result = pd.DataFrame({
    "Coefficient": model.params,
    "t-stat": model.tvalues,
    "p-value": model.pvalues
})
result["Significant (0.01)"] = result["p-value"] < 0.01

In [ ]:
print("\n========== T Test ==========")
print(result)

=====================================================================================
Q2(e) - Parsimonious Model Selection
=====================================================================================

In [ ]:
def regression(vars):
    return sm.OLS(y, sm.add_constant(df_clean[vars])).fit()

In [ ]:
# =====================================================================================
# 1. Correlation Matrix
# =====================================================================================
print("\n========== Q2(e) - Correlation Matrix ==========")
corr = df_clean[predictors].corr()
print(corr.round(4))

In [ ]:
print("\n========== Highly Correlated Pairs (|r| > 0.80) ==========")

In [ ]:
for i in range(len(predictors)):
    for j in range(i + 1, len(predictors)):
        r = corr.iloc[i, j]
        if abs(r) > 0.80: print(predictors[i], "<->", predictors[j], "r =", round(r, 4))
# =====================================================================================
# 2. Initial VIF
# =====================================================================================
def get_vif(vars):
    X = sm.add_constant(df_clean[vars])
    return pd.DataFrame({"Variable": X.columns, "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})

In [ ]:
print("\n========== Initial VIF ==========")
print(get_vif(predictors).round(4))
# =====================================================================================
# 3. Remove Multicollinearity Based on Adjusted R-squared
# =====================================================================================
current = predictors.copy()
# ------------------------------
# Step 01: Avg_Room_Rate_RM vs Competitor_Price_RM
# ------------------------------
print("\n========== Step 01: Avg_Room_Rate_RM vs Competitor_Price_RM ==========")
test1 = current.copy()
test1.remove("Avg_Room_Rate_RM")

In [ ]:
test2 = current.copy()
test2.remove("Competitor_Price_RM")

In [ ]:
model1 = regression(test1)
model2 = regression(test2)

In [ ]:
print("Remove Avg_Room_Rate_RM: Adjusted R-squared =", round(model1.rsquared_adj, 4))
print("Remove Competitor_Price_RM: Adjusted R-squared =", round(model2.rsquared_adj, 4))

In [ ]:
if model1.rsquared_adj > model2.rsquared_adj:
    current = test1
    print("Decision: Remove Avg_Room_Rate_RM")
else:
    current = test2
    print("Decision: Remove Competitor_Price_RM")

In [ ]:
# ------------------------------
# Step 02: Occupancy_Rate_% vs Num_Bookings
# ------------------------------
print("\n========== Step 02: Occupancy_Rate_% vs Num_Bookings ==========")

In [ ]:
test1 = current.copy()
test1.remove("Occupancy_Rate_%")

In [ ]:
test2 = current.copy()
test2.remove("Num_Bookings")

In [ ]:
model1 = regression(test1)
model2 = regression(test2)

In [ ]:
print("Remove Occupancy_Rate_%: Adjusted R-squared =", round(model1.rsquared_adj, 4))
print("Remove Num_Bookings: Adjusted R-squared =", round(model2.rsquared_adj, 4))

In [ ]:
if model1.rsquared_adj > model2.rsquared_adj:
    current = test1
    print("Decision: Remove Occupancy_Rate_%")
else:
    current = test2
    print("Decision: Remove Num_Bookings")

In [ ]:
# ------------------------------
# Step 03: Occupancy_Rate_% vs Staff_Count
# ------------------------------
print("\n========== Step 03: Occupancy_Rate_% vs Staff_Count ==========")

In [ ]:
test1 = current.copy()
test1.remove("Occupancy_Rate_%")

In [ ]:
test2 = current.copy()
test2.remove("Staff_Count")

In [ ]:
model1 = regression(test1)
model2 = regression(test2)

In [ ]:
print("Remove Occupancy_Rate_%: Adjusted R-squared =", round(model1.rsquared_adj, 4))
print("Remove Staff_Count: Adjusted R-squared =", round(model2.rsquared_adj, 4))

In [ ]:
if model1.rsquared_adj > model2.rsquared_adj:
    current = test1
    print("Decision: Remove Occupancy_Rate_%")
else:
    current = test2
    print("Decision: Remove Staff_Count")

In [ ]:
# =====================================================================================
# 4. Regression Before P-value Check
# =====================================================================================
print("\n========== Regression Before P-value Check ==========")

In [ ]:
model = regression(current)

In [ ]:
print("Current predictors:", current)
print("Adjusted R-squared:", round(model.rsquared_adj, 6))

In [ ]:
print(pd.DataFrame({
    "Coefficient": model.params,
    "T-stat": model.tvalues,
    "P-value": model.pvalues
}).round(6))

In [ ]:
# =====================================================================================
# 5. Backward Elimination Based on P-value (α = 0.01)
# =====================================================================================
print("\n========== BACKWARD ELIMINATION (α = 0.01) ==========")

In [ ]:
while True:
    model = regression(current)
    p_values = model.pvalues.drop("const")
    highest_p = p_values.max()
    worst_variable = p_values.idxmax()

    print("\nCurrent predictors:", current)
    print("Adjusted R-squared:", round(model.rsquared_adj, 6))
    print("Highest p-value:", worst_variable, "=", round(highest_p, 6))

    if highest_p <= 0.01:
        print("All remaining variables have p-value <= 0.01.")
        break

    print("Removing:", worst_variable)
    current.remove(worst_variable)

In [ ]:
# =====================================================================================
# 6. Final Parsimonious Model
# =====================================================================================
final_predictors = current.copy()
final_model = regression(final_predictors)

In [ ]:
print("\n========== FINAL PARSIMONIOUS MODEL ==========")

In [ ]:
print("Final significant predictors:", final_predictors)
print("Final Adjusted R-squared:", round(final_model.rsquared_adj, 6))

In [ ]:
print(pd.DataFrame({
    "Coefficient": final_model.params,
    "T-stat": final_model.tvalues,
    "P-value": final_model.pvalues
}).round(6))

In [ ]:
# =====================================================================================
# 7. Final Correlation Matrix
# =====================================================================================
print("\n========== FINAL CORRELATION MATRIX ==========")

In [ ]:
corr_final = df_clean[final_predictors].corr()
print(corr_final.round(4))

In [ ]:
if len(final_predictors) == 2:
    corr_value = corr_final.iloc[0, 1]

    print("\nCorrelation between final predictors:",
          final_predictors[0], "<->", final_predictors[1],
          "=", round(corr_value, 4))

    if abs(corr_value) < 0.80:
        print("No multicollinearity detected (|r| < 0.80).")
    else:
        print("Multicollinearity detected (|r| >= 0.80).")

In [ ]:
# =====================================================================================
# 8. Final VIF
# =====================================================================================
print("\n========== FINAL VIF ==========")

In [ ]:
final_vif = get_vif(final_predictors)
print(final_vif.round(4))
# =====================================================================================
# 9. Conclusion
# =====================================================================================
print("\n========== CONCLUSION ==========")

In [ ]:
max_vif = final_vif.loc[final_vif["Variable"] != "const", "VIF"].max()

In [ ]:
if max_vif < 5:
    print("No evidence of serious multicollinearity exists among the final significant independent variables.")
    print("Therefore, the final regression model is acceptable.")
    print("The remaining independent variables provide independent explanatory information.")
else:
    print("Multicollinearity is detected based on VIF.")
    print("Further corrective action is required.")